# Topic 13 — Random Forest
### Theory → tiny example → from-scratch bagging → sklearn → why the ensemble wins.

A Random Forest is an **ensemble** of many decision trees, each trained slightly differently,
whose predictions are combined by majority vote. The idea: many individually imperfect (and
different) trees average out each other's mistakes and produce a far more robust model than any
single tree.

Two sources of randomness make each tree different:
- **Bagging (Bootstrap Aggregating)**: each tree trains on a random sample of the training data,
  drawn *with replacement* (a "bootstrap sample").
- **Random feature selection**: at each split, each tree only considers a random subset of features,
  not all of them — decorrelating the trees further.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from collections import Counter

rng = np.random.default_rng(0)

## 1. Bootstrap samples

A bootstrap sample is drawn *with replacement*, so it's the same size as the original data, but
some rows appear multiple times and some don't appear at all (~37% left out, on average — these
"out-of-bag" rows are a free validation set forests can use internally).

In [ ]:
original = np.arange(10)   # imagine these are row indices of the training data

bootstrap_sample = rng.choice(original, size=10, replace=True)
print("original indices: ", original)
print("bootstrap sample: ", bootstrap_sample)
print("unique rows used:", len(np.unique(bootstrap_sample)), "out of 10")
print("rows left out ('out-of-bag'):", sorted(set(original) - set(bootstrap_sample)))

## 2. From-scratch mini random forest

Train several decision trees, each on its own bootstrap sample, each considering a random subset
of features at every split — then combine their predictions by majority vote.

In [ ]:
X, y = make_classification(
    n_samples=200, n_features=6, n_informative=4, n_redundant=1,
    n_clusters_per_class=1, class_sep=1.0, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

def train_mini_forest(X, y, n_trees=10, max_features=3, max_depth=4):
    trees = []
    n_samples, n_all_features = X.shape
    for i in range(n_trees):
        # bootstrap sample of rows
        sample_idx = rng.choice(n_samples, size=n_samples, replace=True)
        X_sample, y_sample = X[sample_idx], y[sample_idx]

        # random subset of features for this tree
        feature_idx = rng.choice(n_all_features, size=max_features, replace=False)

        tree = DecisionTreeClassifier(max_depth=max_depth, random_state=i)
        tree.fit(X_sample[:, feature_idx], y_sample)
        trees.append((tree, feature_idx))
    return trees

def predict_mini_forest(trees, X):
    all_preds = np.array([tree.predict(X[:, feat_idx]) for tree, feat_idx in trees])   # shape (n_trees, n_samples)
    # majority vote across trees, for each sample
    final_preds = []
    for col in all_preds.T:
        final_preds.append(Counter(col).most_common(1)[0][0])
    return np.array(final_preds)

forest = train_mini_forest(X_train, y_train, n_trees=15, max_features=3, max_depth=4)
y_pred_forest = predict_mini_forest(forest, X_test)
print("mini forest test accuracy:", accuracy_score(y_test, y_pred_forest))

# Compare against a single tree trained the normal way
single_tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
print("single tree test accuracy:", accuracy_score(y_test, single_tree.predict(X_test)))

## 3. sklearn's RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,     # number of trees
    max_depth=4,
    max_features="sqrt",  # each split considers sqrt(n_features) features -- a common default
    random_state=42
)
rf.fit(X_train, y_train)

print("train accuracy:", accuracy_score(y_train, rf.predict(X_train)))
print("test accuracy:", accuracy_score(y_test, rf.predict(X_test)))

# Feature importance: how much each feature contributed across all trees, on average
importances = rf.feature_importances_
for i, imp in enumerate(importances):
    print(f"feature_{i}: importance={imp:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(range(len(importances)), importances)
plt.xlabel("feature index")
plt.ylabel("importance")
plt.title("Random Forest feature importances")
plt.show()

## 4. Why the ensemble is more robust: variance reduction

A single decision tree is high-variance (Topic 5/12) — retrain it on slightly different data and
you can get a very different tree. Averaging many such trees cancels out a lot of that noise.

In [ ]:
n_trials = 20
single_tree_accs = []
forest_accs = []

for trial in range(n_trials):
    idx = rng.choice(len(X_train), size=len(X_train), replace=True)   # a new resampled training set
    Xt, yt = X_train[idx], y_train[idx]

    st = DecisionTreeClassifier(max_depth=4, random_state=trial).fit(Xt, yt)
    single_tree_accs.append(accuracy_score(y_test, st.predict(X_test)))

    rft = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=trial).fit(Xt, yt)
    forest_accs.append(accuracy_score(y_test, rft.predict(X_test)))

print(f"single tree: mean={np.mean(single_tree_accs):.3f}, std={np.std(single_tree_accs):.3f}")
print(f"random forest: mean={np.mean(forest_accs):.3f}, std={np.std(forest_accs):.3f}")
# The forest usually has a HIGHER mean accuracy AND a LOWER std (more consistent across retrains).

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. In the from-scratch mini forest, change n_trees to 1 and 50 -- how does test accuracy change?
# 2. Change max_features in RandomForestClassifier from "sqrt" to None (uses ALL features every split) --
#    does accuracy or feature-importance spread change?
# 3. Plot single_tree_accs and forest_accs from part 4 as two box plots side by side
#    (hint: reuse the box plot code style from Topic 3) to visually compare their variance.
# 4. Why might using n_estimators=500 instead of 100 give diminishing returns on accuracy
#    while still costing much more compute? (One sentence.)

---
### Next up: **Topic 14 — Support Vector Machines** (very relevant to text classification).

Say "next" when you're ready.